[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tymworld/DROPPS/blob/v1.0.0/examples/colab/01_DROPPS_1_0_Model_and_System_Builder.ipynb)

# DROPPS 1.0 Molecular and System Builder

Build a compact **cubic** box containing many copies of one protein for the
phase-coexistence workflow. Multi-component systems remain available as an
extension. Optional PTMs, angle restraints, elastic networks, residue
modifications, and deterministic packing match the construction workflow
described in the manuscript.

Do not elongate the initial box here. The simulation notebook first condenses
this compact system under NPT, then expands one axis tenfold and starts the NVT
slab-production stage.

All operations use the public `dps` CLI. Bead IDs in angle and elastic input
files are **1-based**; DROPPS index groups used for trajectory analysis are
**0-based** and are handled in later notebooks.


In [ ]:
#@title Install DROPPS 1.0
#@markdown The default route installs the immutable wheel attached to the
#@markdown public GitHub `v1.0.0` release. Upload remains available for offline use.
installation_source = "Install from the DROPPS v1.0.0 GitHub release" #@param ["Install from the DROPPS v1.0.0 GitHub release", "Upload the DROPPS 1.0 wheel", "Install from another wheel URL"]
wheel_url = "https://github.com/tymworld/DROPPS/releases/download/v1.0.0/dropps-1.0-py3-none-any.whl" #@param {type:"string"}
accelerator_dependencies = "auto" #@param ["auto", "cuda12", "cuda13", "latest"]

import importlib.metadata
import re
import shutil
import subprocess
import sys
from pathlib import Path


def detect_install_extra():
    if accelerator_dependencies != "auto":
        return accelerator_dependencies
    if shutil.which("nvidia-smi"):
        probe = subprocess.run(
            ["nvidia-smi"], text=True, capture_output=True, check=False
        ).stdout
        match = re.search(r"CUDA Version:\s*(\d+)", probe)
        if match and int(match.group(1)) >= 13:
            return "cuda13"
        return "cuda12"
    return "latest"


if installation_source.startswith("Upload"):
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError(
            "This upload form is intended for Google Colab. Set installation_source "
            "to the URL option when running elsewhere."
        ) from exc
    uploaded = files.upload()
    wheel_candidates = [Path(name) for name in uploaded if name.endswith(".whl")]
    if len(wheel_candidates) != 1:
        raise ValueError("Upload exactly one DROPPS 1.0 .whl file.")
    wheel_target = str(wheel_candidates[0].resolve())
else:
    if not wheel_url.strip():
        raise ValueError("Provide the published DROPPS 1.0 wheel URL.")
    wheel_target = wheel_url.strip()

extra = detect_install_extra()
if wheel_target.startswith(("https://", "http://")):
    package_spec = f"dropps[{extra}] @ {wheel_target}"
else:
    package_spec = f"{wheel_target}[{extra}]"
print(f"Installing {package_spec}")
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--upgrade",
        "--upgrade-strategy",
        "only-if-needed",
        package_spec,
    ],
    check=True,
)

version = importlib.metadata.version("dropps")
if version != "1.0":
    raise RuntimeError(f"Expected DROPPS 1.0, but installed {version}.")
subprocess.run(["dps", "--version"], check=True)


In [ ]:
#@title Shared notebook helpers
import hashlib
import json
import os
import shlex
import subprocess
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

COMMAND_LOG = []


def run_command(arguments, *, cwd=None, input_text=None, check=True):
    command = [str(value) for value in arguments]
    print("$", shlex.join(command))
    result = subprocess.run(
        command,
        cwd=None if cwd is None else str(cwd),
        input=input_text,
        text=True,
        capture_output=True,
        check=False,
    )
    if result.stdout:
        print(result.stdout, end="" if result.stdout.endswith("\n") else "\n")
    if result.stderr:
        print(result.stderr, end="" if result.stderr.endswith("\n") else "\n")
    COMMAND_LOG.append(
        {
            "time_utc": datetime.now(timezone.utc).isoformat(),
            "cwd": str(Path(cwd or Path.cwd()).resolve()),
            "command": command,
            "returncode": result.returncode,
        }
    )
    if check and result.returncode:
        raise RuntimeError(
            f"Command failed with exit code {result.returncode}: {shlex.join(command)}"
        )
    return result


def dps(*arguments, cwd=None, input_text=None, check=True):
    return run_command(
        ["dps", *arguments], cwd=cwd, input_text=input_text, check=check
    )


def reset_task_directory(path):
    path = Path(path).resolve()
    if not path.name.startswith("dropps_"):
        raise ValueError(f"Refusing to reset unexpected directory: {path}")
    if path.exists():
        import shutil

        shutil.rmtree(path)
    path.mkdir(parents=True)
    return path


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def safe_extract_zip(archive, destination):
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as bundle:
        for member in bundle.infolist():
            target = (destination / member.filename).resolve()
            if destination not in target.parents and target != destination:
                raise ValueError(f"Unsafe ZIP member: {member.filename}")
        bundle.extractall(destination)


def find_unique(root, basename):
    matches = [path for path in Path(root).rglob(basename) if path.is_file()]
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected one file named {basename!r} below {root}, found {len(matches)}."
        )
    return matches[0]


def make_zip(paths, output_path, *, base=None):
    output_path = Path(output_path)
    base = Path(base or output_path.parent).resolve()
    with zipfile.ZipFile(output_path, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
        for source in sorted({Path(path).resolve() for path in paths}):
            if source == output_path.resolve() or not source.is_file():
                continue
            try:
                arcname = source.relative_to(base)
            except ValueError:
                arcname = Path(source.name)
            bundle.write(source, arcname)
    print(f"Wrote {output_path} ({output_path.stat().st_size / 1024:.1f} KiB)")
    return output_path


def download(path):
    try:
        from google.colab import files
    except ImportError:
        print(f"Result available at {Path(path).resolve()}")
    else:
        files.download(str(path))


def optional_time_arguments(start, end, interval):
    arguments = []
    for option, value in (("-b", start), ("-e", end), ("-dt", interval)):
        if str(value).strip():
            arguments.extend([option, str(value).strip()])
    return arguments


def read_xvg(path):
    legends = {}
    metadata = {}
    rows = []
    with Path(path).open(encoding="utf-8") as stream:
        for raw in stream:
            line = raw.strip()
            if not line or line.startswith("#"):
                continue
            if line.startswith("@"):
                re_module = __import__("re")
                match = re_module.search(r's(\d+)\s+legend\s+"(.*)"', line)
                if match:
                    legends[int(match.group(1)) + 1] = match.group(2)
                for key, pattern in {
                    "title": r'@\s+title\s+"(.*)"',
                    "xlabel": r'@\s+xaxis\s+label\s+"(.*)"',
                    "ylabel": r'@\s+yaxis\s+label\s+"(.*)"',
                }.items():
                    label_match = re_module.search(pattern, line)
                    if label_match:
                        metadata[key] = label_match.group(1)
                continue
            rows.append([float(value) for value in line.split()])
    data = np.asarray(rows, dtype=float)
    if data.ndim != 2 or data.shape[1] < 2:
        raise ValueError(f"No plottable data in {path}")
    return data, legends, metadata


def plot_xvg(path, *, title=None, output=None):
    data, legends, metadata = read_xvg(path)
    fig, ax = plt.subplots(figsize=(7.2, 4.2))
    for column in range(1, data.shape[1]):
        ax.plot(data[:, 0], data[:, column], label=legends.get(column, f"series {column}"))
    ax.set_title(title or metadata.get("title", Path(path).stem))
    ax.set_xlabel(metadata.get("xlabel", "x / time"))
    ax.set_ylabel(metadata.get("ylabel", "value"))
    if data.shape[1] > 2:
        ax.legend(frameon=False, fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    output_path = Path(output) if output is not None else Path(path).with_suffix(".png")
    fig.savefig(output_path, dpi=220, bbox_inches="tight")
    print(f"Saved figure: {output_path}")
    plt.show()
    return fig, output_path


## 1 Define up to three molecular components

The default is a lightweight 80-copy, single-protein construction example,
following the manuscript's 5×5×5 placement grid. Replace the sequence with
the target protein for a scientific run. Mapping uses Cα coordinates from the
first model of an uploaded all-atom PDB. Enable Components B/C only for a
co-phase-separation system and use compatible force-field families.


In [ ]:
#@title Component A
component_a_enabled = True #@param {type:"boolean"}
component_a_name = "SCAFFOLD" #@param {type:"string"}
component_a_sequence = "FWFWFWFWFWFWFWFW" #@param {type:"string"}
component_a_count = 80 #@param {type:"integer"}
component_a_forcefield = "HPS" #@param ["HPS", "HPST", "CALVADOS2", "HPSRNA", "MPiPi", "MPiPi_PTM"]
component_a_input_pdb = "" #@param {type:"string"}
component_a_ptms = "" #@param {type:"string"}
component_a_radius_nm = 2.0 #@param {type:"number"}
component_a_extension = 0.5 #@param {type:"slider", min:0, max:1, step:0.05}
component_a_residue_index = 1 #@param {type:"integer"}
component_a_charge_ntd = False #@param {type:"boolean"}
component_a_charge_ctd = False #@param {type:"boolean"}


In [ ]:
#@title Component B
component_b_enabled = False #@param {type:"boolean"}
component_b_name = "CLIENT" #@param {type:"string"}
component_b_sequence = "KKGGEEDD" #@param {type:"string"}
component_b_count = 4 #@param {type:"integer"}
component_b_forcefield = "HPS" #@param ["HPS", "HPST", "CALVADOS2", "HPSRNA", "MPiPi", "MPiPi_PTM"]
component_b_input_pdb = "" #@param {type:"string"}
component_b_ptms = "" #@param {type:"string"}
component_b_radius_nm = 2.0 #@param {type:"number"}
component_b_extension = 0.5 #@param {type:"slider", min:0, max:1, step:0.05}
component_b_residue_index = 1 #@param {type:"integer"}
component_b_charge_ntd = False #@param {type:"boolean"}
component_b_charge_ctd = False #@param {type:"boolean"}


In [ ]:
#@title Component C
component_c_enabled = False #@param {type:"boolean"}
component_c_name = "RNA" #@param {type:"string"}
component_c_sequence = "AAAAAAAAAA" #@param {type:"string"}
component_c_count = 2 #@param {type:"integer"}
component_c_forcefield = "HPSRNA" #@param ["HPS", "HPST", "CALVADOS2", "HPSRNA", "MPiPi", "MPiPi_PTM"]
component_c_input_pdb = "" #@param {type:"string"}
component_c_ptms = "" #@param {type:"string"}
component_c_radius_nm = 2.0 #@param {type:"number"}
component_c_extension = 0.5 #@param {type:"slider", min:0, max:1, step:0.05}
component_c_residue_index = 1 #@param {type:"integer"}
component_c_charge_ntd = False #@param {type:"boolean"}
component_c_charge_ctd = False #@param {type:"boolean"}


In [ ]:
#@title Upload optional all-atom PDB reference structures
upload_reference_pdbs = False #@param {type:"boolean"}
random_seed = 1215 #@param {type:"integer"}

BASE = Path("/content") if Path("/content").is_dir() else Path.cwd()
WORKDIR = reset_task_directory(BASE / "dropps_model_builder")
UPLOAD_DIR = WORKDIR / "uploads"
UPLOAD_DIR.mkdir()

if upload_reference_pdbs:
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError("Upload reference structures in Google Colab.") from exc
    import shutil

    for name in files.upload():
        source = Path(name)
        if source.suffix.lower() != ".pdb":
            raise ValueError(f"Expected PDB input, received {source}")
        shutil.copy2(source, UPLOAD_DIR / source.name)

print("Working directory:", WORKDIR)


In [ ]:
#@title Build monomer structures and self-contained ITP topologies
import re

def component(prefix):
    scope = globals()
    return {
        "enabled": scope[f"component_{prefix}_enabled"],
        "name": scope[f"component_{prefix}_name"].strip(),
        "sequence": scope[f"component_{prefix}_sequence"].strip(),
        "count": int(scope[f"component_{prefix}_count"]),
        "forcefield": scope[f"component_{prefix}_forcefield"],
        "input_pdb": scope[f"component_{prefix}_input_pdb"].strip(),
        "ptms": scope[f"component_{prefix}_ptms"].strip(),
        "radius": float(scope[f"component_{prefix}_radius_nm"]),
        "extension": float(scope[f"component_{prefix}_extension"]),
        "residue_index": int(scope[f"component_{prefix}_residue_index"]),
        "charge_ntd": scope[f"component_{prefix}_charge_ntd"],
        "charge_ctd": scope[f"component_{prefix}_charge_ctd"],
    }

components = [item for item in map(component, "abc") if item["enabled"]]
if not components:
    raise ValueError("Enable at least one component.")
if len({item["name"] for item in components}) != len(components):
    raise ValueError("Component names must be unique.")

for item in components:
    if not re.fullmatch(r"[A-Za-z][A-Za-z0-9_]*", item["name"]):
        raise ValueError(f"Use a simple alphanumeric molecule name: {item['name']!r}")
    if not item["sequence"]:
        raise ValueError(f"Sequence is empty for {item['name']}")
    arguments = [
        "pdb2dps", "-s", item["sequence"], "-ff", item["forcefield"],
        "-oc", f"{item['name']}.pdb", "-op", f"{item['name']}.itp",
        "-on", item["name"], "-ri", item["residue_index"],
        "-r", item["radius"], "-e", item["extension"],
        "--seed", random_seed,
    ]
    if item["input_pdb"]:
        source = Path(item["input_pdb"])
        if not source.is_file():
            source = UPLOAD_DIR / item["input_pdb"]
        if not source.is_file():
            raise FileNotFoundError(source)
        arguments.extend(["-f", source])
    if item["ptms"]:
        arguments.extend(["-ptm", *item["ptms"].split()])
    if item["charge_ntd"]:
        arguments.append("-cNTD")
    if item["charge_ctd"]:
        arguments.append("-cCTD")
    dps(*arguments, cwd=WORKDIR)
    item["pdb"] = WORKDIR / f"{item['name']}.pdb"
    item["itp"] = WORKDIR / f"{item['name']}.itp"

print("Built:", ", ".join(item["name"] for item in components))


## 2 Optional topology refinements

Enter semicolon-separated records. Angle records use
`CENTER_RESIDUE ANGLE_DEG FORCE_CONSTANT`. Elastic groups use compact
ranges such as `1-77; 106-177; 192-256`; the notebook expands them into
the explicit 1-based bead lists required by `genelastic`.


In [ ]:
#@title Angle restraints with dps addangle
angle_target = "SCAFFOLD" #@param {type:"string"}
angle_records = "" #@param {type:"string"}

if angle_records.strip():
    target = next(item for item in components if item["name"] == angle_target)
    angle_file = WORKDIR / f"{angle_target}_angles.dat"
    angle_file.write_text(
        "\n".join(part.strip() for part in angle_records.split(";") if part.strip()) + "\n",
        encoding="utf-8",
    )
    output = WORKDIR / f"{angle_target}_angles.itp"
    dps(
        "addangle", "-ip", target["itp"], "-op", output,
        "-al", angle_file, cwd=WORKDIR,
    )
    target["itp"] = output
else:
    print("No angle restraints requested.")


In [ ]:
#@title Elastic networks with dps genelastic
elastic_target = "SCAFFOLD" #@param {type:"string"}
elastic_groups = "" #@param {type:"string"}
elastic_force_constant = 5000.0 #@param {type:"number"}
elastic_lower_nm = 0.5 #@param {type:"number"}
elastic_upper_nm = 0.9 #@param {type:"number"}

def expand_integer_ranges(expression):
    values = []
    for token in expression.replace(",", " ").split():
        if "-" in token:
            start, stop = map(int, token.split("-", 1))
            if stop < start:
                raise ValueError(f"Descending range: {token}")
            values.extend(range(start, stop + 1))
        else:
            values.append(int(token))
    return values

if elastic_groups.strip():
    target = next(item for item in components if item["name"] == elastic_target)
    lines = []
    for group in elastic_groups.split(";"):
        if group.strip():
            lines.append(" ".join(map(str, expand_integer_ranges(group))))
    elastic_file = WORKDIR / f"{elastic_target}_elastic.dat"
    elastic_file.write_text("\n".join(lines) + "\n", encoding="utf-8")
    output = WORKDIR / f"{elastic_target}_elastic.itp"
    dps(
        "genelastic", "-f", target["pdb"], "-p", target["itp"],
        "-o", output, "-er", elastic_file,
        "-ef", elastic_force_constant, "-el", elastic_lower_nm,
        "-eu", elastic_upper_nm, cwd=WORKDIR,
    )
    target["itp"] = output
else:
    print("No elastic network requested.")


In [ ]:
#@title Optional post-build residue modification with dps modifyres
apply_residue_modifications = False #@param {type:"boolean"}
modification_target = "SCAFFOLD" #@param {type:"string"}
residue_modifications = "S3SMP" #@param {type:"string"}
modification_forcefield = "MPiPi_PTM" #@param ["HPS", "HPST", "CALVADOS2", "HPSRNA", "MPiPi", "MPiPi_PTM"]

if apply_residue_modifications:
    target = next(item for item in components if item["name"] == modification_target)
    modified_pdb = WORKDIR / f"{modification_target}_modified.pdb"
    modified_itp = WORKDIR / f"{modification_target}_modified.itp"
    dps(
        "modifyres", "-ip", target["itp"], "-if", target["pdb"],
        "-op", modified_itp, "-of", modified_pdb,
        "-ff", modification_forcefield,
        "-m", *residue_modifications.split(), cwd=WORKDIR,
    )
    target["pdb"], target["itp"] = modified_pdb, modified_itp
else:
    print("No post-build residue modification requested.")


## 3 Pack the compact cubic simulation box


In [ ]:
#@title System packing parameters
mesh_x = 5 #@param {type:"integer"}
mesh_y = 5 #@param {type:"integer"}
mesh_z = 5 #@param {type:"integer"}
minimum_gap_nm = 1.0 #@param {type:"number"}
box_type = "cubic" #@param ["cubic", "anisotropy", "xy"]
minimum_x_nm = 5.0 #@param {type:"number"}
minimum_y_nm = 5.0 #@param {type:"number"}
minimum_z_nm = 5.0 #@param {type:"number"}
shuffle_components = True #@param {type:"boolean"}
non_cubic_molecule = False #@param {type:"boolean"}

total_count = sum(item["count"] for item in components)
if mesh_x * mesh_y * mesh_z < total_count:
    raise ValueError("The mesh has fewer sites than requested molecules.")

arguments = ["genmesh", "-f"]
arguments.extend(item["pdb"] for item in components)
arguments.append("-p")
arguments.extend(item["itp"] for item in components)
arguments.append("-n")
arguments.extend(item["count"] for item in components)
arguments.extend(
    [
        "-mesh", mesh_x, mesh_y, mesh_z,
        "-g", minimum_gap_nm, "-bt", box_type,
        "-mx", minimum_x_nm, "-my", minimum_y_nm, "-mz", minimum_z_nm,
        "--seed", random_seed, "-oc", "system.pdb", "-op", "system.top",
    ]
)
if shuffle_components:
    arguments.append("-s")
if non_cubic_molecule:
    arguments.append("-ncm")
dps(*arguments, cwd=WORKDIR)
SYSTEM_PDB = WORKDIR / "system.pdb"
SYSTEM_TOP = WORKDIR / "system.top"


The exported `system.pdb` is deliberately compact and cubic. Its box must be
elongated only after the NPT stage has produced a condensed `npt.pdb`.


## 4 Inspect and export the model bundle


In [ ]:
#@title Preview bead coordinates
def pdb_coordinates_nm(path):
    rows = []
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        if line.startswith(("ATOM  ", "HETATM")):
            rows.append([float(line[30:38]), float(line[38:46]), float(line[46:54])])
    return np.asarray(rows) / 10.0

coordinates = pdb_coordinates_nm(SYSTEM_PDB)
fig = plt.figure(figsize=(6.5, 5.5))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(*coordinates.T, s=8, alpha=0.7)
ax.set(xlabel="x (nm)", ylabel="y (nm)", zlabel="z (nm)", title="Initial system")
fig.tight_layout()
plt.show()


In [ ]:
#@title Save manifest and download
selected_files = [SYSTEM_PDB, SYSTEM_TOP]
for item in components:
    selected_files.extend([item["pdb"], item["itp"]])
selected_files = sorted({Path(path).resolve() for path in selected_files})
manifest = {
    "notebook": "01_DROPPS_1_0_Model_and_System_Builder.ipynb",
    "dropps_version": "1.0",
    "seed": random_seed,
    "components": [
        {
            key: value
            for key, value in item.items()
            if key not in {"pdb", "itp", "input_pdb"}
        }
        | {"pdb": item["pdb"].name, "itp": item["itp"].name}
        for item in components
    ],
    "system_pdb": SYSTEM_PDB.name,
    "system_top": SYSTEM_TOP.name,
    "files": {path.name: sha256(path) for path in selected_files},
    "commands": COMMAND_LOG,
}
manifest_path = WORKDIR / "model_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
selected_files.append(manifest_path)
archive = make_zip(
    selected_files, WORKDIR / "DROPPS_1_0_model_bundle.zip", base=WORKDIR
)
download(archive)
